# Notebook 05 - Compare Base / Raw SFT / Adapted SFT

This notebook generates comparable responses for the three model conditions and builds a qualitative comparison table.

The important experimental discipline: use the same prompts and sampling settings for all three models.

**Files read:**
- [`../configs/project.yaml`](../configs/project.yaml) - model registry, sampling settings, prompt path, and response output paths.
- [`../configs/language_adherence_prompts.jsonl`](../configs/language_adherence_prompts.jsonl) - short Kirundi prompts used for the qualitative comparison.
- [`../results/models/sft_raw/checkpoints.jsonl`](../results/models/sft_raw/checkpoints.jsonl) - final sampler reference from Notebook 03 for the raw-data SFT model.
- [`../results/models/sft_adapted/checkpoints.jsonl`](../results/models/sft_adapted/checkpoints.jsonl) - final sampler reference from Notebook 04 for the adapted-data SFT model.

**Files written:**
- [`../results/responses/base.jsonl`](../results/responses/base.jsonl) - base-model responses to the shared prompts.
- [`../results/responses/sft_raw.jsonl`](../results/responses/sft_raw.jsonl) - raw-data SFT model responses to the shared prompts.
- [`../results/responses/sft_adapted.jsonl`](../results/responses/sft_adapted.jsonl) - adapted-data SFT model responses to the shared prompts.


In [ ]:
import json
import sys
from pathlib import Path

In [ ]:
def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'configs' / 'project.yaml').exists():
            return candidate
    raise FileNotFoundError(
        'Run this notebook from inside the adaption-kirundi-sft-starter repo.'
    )

ROOT = find_repo_root(Path.cwd())

sys.path.append(str(ROOT / 'src'))

from kirundi_sft_starter.tinker_utils import generate_model_responses
from kirundi_sft_starter.utils import load_yaml, read_jsonl

project_config = load_yaml(ROOT / 'configs/project.yaml')

In [ ]:
print(json.dumps(project_config['models']['registry'], indent=2))

## Generate comparable responses

This cell calls Tinker directly from the notebook and writes the standard response files under `results/responses/`. Run Notebooks 03 and 04 first so the two SFT checkpoint files exist.

In [ ]:
comparison_prompts = read_jsonl(ROOT / project_config['comparison']['prompts_path'])

for model_key in project_config['models']['registry']:
    response_path = generate_model_responses(project_config, model_key, comparison_prompts)
    print(f'Wrote {response_path}')

In [15]:
responses_by_model = {}

for model_key, model_config in project_config['models']['registry'].items():
    response_path = ROOT / model_config['response_path']
    if response_path.exists():
        responses_by_model[model_key] = {
            row['prompt_id']: row
            for row in read_jsonl(response_path)
        }
    else:
        responses_by_model[model_key] = {}
        print(f'Missing response file for {model_key}: {response_path}')

In [16]:
model_labels = {
    'base': 'Base',
    'sft_raw': 'SFT no Adaption',
    'sft_adapted': 'SFT with Adaption',
}

for prompt_number, prompt_row in enumerate(comparison_prompts, start=1):
    prompt_id = prompt_row['prompt_id']
    print('=' * 100)
    print(f'Prompt #{prompt_number} ({prompt_id})')
    print(prompt_row['prompt'])
    print()

    for model_key in ['base', 'sft_raw', 'sft_adapted']:
        print(f'{model_labels[model_key]}:')
        response_row = responses_by_model.get(model_key, {}).get(prompt_id)
        if response_row is None:
            print('[missing response]')
        else:
            print(response_row.get('response', '[missing response]'))
        print()

Prompt #1 (lang_001)
Subiza mu Kirundi: Ni gute umuntu yokwiga ikoranabuhanga intambwe ku yindi?

Base:
Kugera mu Kirundi, "Subiza mu Kirundi: Ni gute umuntu yokwiga ikoranabuhanga intambwe ku yindi?" ni "Subiza mu Kirundi: Ni gute umuntu yokwiga ikoranabuhanga intambwe ku yindi?"

SFT no Adaption:
Umuntu yoroshye ikoranabuhanga intambwe ni umwanya w’inyigisho (umwanya w’inyigisho) w’uburezi bw’ikoranabuhanga. Iyi umwanya w’inyigisho yongera uburezi bw’ikoranabuhanga, yongere uburezi bw’uburezi, n’uburezi bw’uburezi bw’uburezi. Iyi umwanya w’inyigisho yongere uburezi bw’ikoranabuhanga, yongere uburezi bw’uburezi, n’uburezi bw’uburezi bw’uburezi. Ni umwanya w’inyigisho w’ikoranabuhanga.

SFT with Adaption:
Ni umuntu w’umwanya w’ikoranabuhanga intambwe, ariko ari umwanya w’umwanya w’ikoranabuhanga intambwe (umwanya w’umwanya). Aya umwanya w’umwanya w’ikoranabuhanga intambwe, ariko ari umwanya w’umwanya w’ikoranabuhanga intambwe (umwanya w’umwanya). Aya umwanya w’umwanya w’ikoranabuhanga 

## Qualitative review questions

- Did the answer stay in Kirundi?
- Did it answer the question directly?
- Did it copy unwanted reasoning traces?
- Did Adaption appear to improve clarity or formatting?
- Are there examples that need native speaker review before any conclusion?

### Preliminary qualitative analysis

The Base model mostly copies prompts or falls into repetition loops, while the SFT models produce more answer-shaped text without consistently preserving meaning. This review is preliminary and LLM-assisted rather than based on native-speaker evaluation, so it should be treated as directional. In this checked-in run, the SFT models were trained on only 200 samples, so weak performance is expected at this stage. Still, the Adaption version tends to start closer to the requested task and preserves more relevant vocabulary, suggesting that Adaption may already be helping with task framing or lexical alignment. With more data, training time, and native-speaker validation, this could become a stronger signal for improving full answer quality.